## Notebook script to create a pMHC single-chain trimer
Create a tool that is able to create the nucleotide sequence to insert into desired destination vector. With this tool, you should be able to insert a peptide sequence and select the HLA allele and it will generate the nucleotide sequence you would order for Gibson

Sequence is 5addon-SP-Peptide-L1-B2M-L2-HLA-3addon.

In [ ]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../tools")

import pandas as pd
from config import ALL_DATA_DIR

In [ ]:
# dictionary for linkers
linker_nuc = {
    'L1' : "ggtggaggaggttctggaggtggtggtagtggtggtggtggttcc",
    'L2' : "ggtggtggtggtagtggtggtggtggttcaggtggtggtggttccggtggtggtggttcc"
}

linker_aa = {
    'L1': "GGGGSGGGGSGGGGS",
    'L2': "GGGGSGGGGSGGGGSGGGGS"
}

B2M_aa = "IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"

# dictionary for destination vectors
destination_vector_nuc = {
    "pHIV-EGFP" : {
        "5_addon": "tcgtgagcggccgctgagtt",
        "3_addon": "aactattctagagtacccgggctaggatcc"},
    "pHIV-dTomato": {
        "5_addon": "gaactgaattcatcgacgtt",
        "3_addon": "aactattctagagtacccgggctaggatcc"}
}

# leader signal peptide
leader_peptide_aa = {
    "B2M": "MSRSVALAVLALLSLSGLEA",
    "HGH": "MATGSRTSLLLAFGLLCLPWLQEGSA"
}

HLA_aa = {
    "HLA-A201_BM": "HSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGAYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSS",
    "HLA-A201_RS": "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGAYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSS",
}

HLA_EndDomains = {
    "Transmembrane": {
        "HLA-A": "PSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSS",
        "HLA-B": "PSSQSTVPIVGIVAGLAVLAVVVIGAVVAAVMCR"
    }
}

HLA_nuc = {
    "HLA-A2_BM": "",
    "HLA-A2_RS": ""
}

peptide_nuc = {
    "MART-1": "gaggccgcggggataggtatattgacggta",
    "MART-2": "mart2",
    "MART-3": "mart3",
}

### Verifying the HLA-A2 used

In [ ]:
# two 0201 alleles found 
HLAA2010101 = "MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSSDRKGGSYSQAASSDSAQGSDVSLTACKV"
HLAA2010201 = "MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSSDRKGGSYSQAASSDSAQGSDVSLTACKV"

In [ ]:
HLAA2010101 == HLAA2010201

True

### Comparing the HLA sequence used in MB vs RS

In [ ]:
HLAA2_MB = "HSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGAYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPS"
HLAA2_aa_RS = "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGAYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE"


In [ ]:
# removing the leader peptide
HLAA2010101[24:]
HLAA2_cutleader = HLAA2010101[26:]

'GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSSDRKGGSYSQAASSDSAQGSDVSLTACKV'

HLA-A2 from Roland Strong's paper has a different sequence that the one used by Michael Birnbaum. 

GC is included in RS but for MB the GS is a part of the L2 linker:
```
RS: GGGGSGSHS
MB: GGGGSHS
```

In [ ]:
# exon 5 variations
HLAA2_full_genomic = "CAGAAGCAGAGGGGTCAGGGCGAAGTCCCAGGGCCCCAGGCGTGGCTCTCAGGGTCTCAGGCCCCGAAGGCGGTGTATGGATTGGGGAGTCCCAGCCTTGGGGATTCCCCAACTCCGCAGTTTCTTTTCTCCCTCTCCCAACCTATGTAGGGTCCTTCTTCCTGGATACTCACGACGCGGACCCAGTTCTCACTCCCATTGGGTGTCGGGTTTCCAGAGAAGCCAATCAGTGTCGTCGCGGTCGCGGTTCTAAAGTCCGCACGCACCCACCGGGACTCAGATTCTCCCCAGACGCCGAGGATGGCCGTCATGGCGCCCCGAACCCTCGTCCTGCTACTCTCGGGGGCTCTGGCCCTGACCCAGACCTGGGCGGGTGAGTGCGGGGTCGGGAGGGAAACGGCCTCTGTGGGGAGAAGCAACGGGCCCGCCTGGCGGGGGCGCAGGACCCGGGAAGCCGCGCCGGGAGGAGGGTCGGGCGGGTCTCAGCCACTCCTCGTCCCCAGGCTCTCACTCCATGAGGTATTTCTTCACATCCGTGTCCCGGCCCGGCCGCGGGGAGCCCCGCTTCATCGCAGTGGGCTACGTGGACGACACGCAGTTCGTGCGGTTCGACAGCGACGCCGCGAGCCAGAGGATGGAGCCGCGGGCGCCGTGGATAGAGCAGGAGGGTCCGGAGTATTGGGACGGGGAGACACGGAAAGTGAAGGCCCACTCACAGACTCACCGAGTGGACCTGGGGACCCTGCGCGGCTACTACAACCAGAGCGAGGCCGGTGAGTGACCCCGGCCCGGGGCGCAGGTCACGACCTCTCATCCCCCACGGACGGGCCAGGTCGCCCACAGTCTCCGGGTCCGAGATCCGCCCCGAAGCCGCGGGACCCCGAGACCCTTGCCCCGGGAGAGGCCCAGGCGCCTTTACCCGGTTTCATTTTCAGTTTAGGCCAAAAATCCCCCCAGGTTGGTCGGGGCGGGGCGGGGCTCGGGGGACCGGGCTGACCGCGGGGTCCGGGCCAGGTTCTCACACCGTCCAGAGGATGTATGGCTGCGACGTGGGGTCGGACTGGCGCTTCCTCCGCGGGTACCACCAGTACGCCTACGACGGCAAGGATTACATCGCCCTGAAAGAGGACCTGCGCTCTTGGACCGCGGCGGACATGGCAGCTCAGACCACCAAGCACAAGTGGGAGGCGGCCCATGTGGCGGAGCAGTTGAGAGCCTACCTGGAGGGCACGTGCGTGGAGTGGCTCCGCAGATACCTGGAGAACGGGAAGGAGACGCTGCAGCGCACGGGTACCAGGGGCCACGGGGCGCCTCCCTGATCGCCTGTAGATCTCCCGGGCTGGCCTCCCACAAGGAGGGGAGACAATTGGGACCAACACTAGAATATCGCCCTCCCTCTGGTCCTGAGGGAGAGGAATCCTCCTGGGTTTCCAGATCCTGTACCAGAGAGTGACTCTGAGGTTCCGCCCTGCTCTCTGACACAATTAAGGGATAAAATCTCTGAAGGAATGACGGGAAGACGATCCCTCGAATACTGATGAGTGGTTCCCTTTGACACACACAGGCAGCAGCCTTGGGCCCGTGACTTTTCCTCTCAGGCCTTGTTCTCTGCTTCACACTCAATGTGTGTGGGGGTCTGAGTCCAGCACTTCTGAGTCCTTCAGCCTCCACTCAGGTCAGGACCAGAAGTCGCTGTTCCCTCTTCAGGGACTAGAATTTTCCACGGAATAGGAGATTATCCCAGGTGCCTGTGTCCAGGCTGGTGTCTGGGTTCTGTGCTCCCTTCCCCATCCCAGGTGTCCTGTCCATTCTCAAGATAGCCACATGTGTGCTGGAGGAGTGTCCCATGACAGATGCAAAATGCCTGAATGATCTGACTCTTCCTGACAGACGCCCCCAAAACGCATATGACTCACCACGCTGTCTCTGACCATGAAGCCACCCTGAGGTGCTGGGCCCTGAGCTTCTACCCTGCGGAGATCACACTGACCTGGCAGCGGGATGGGGAGGACCAGACCCAGGACACGGAGCTCGTGGAGACCAGGCCTGCAGGGGATGGAACCTTCCAGAAGTGGGCGGCTGTGGTGGTGCCTTCTGGACAGGAGCAGAGATACACCTGCCATGTGCAGCATGAGGGTTTGCCCAAGCCCCTCACCCTGAGATGGGGTAAGGAGGGAGACGGGGGTGTCATGTCTTTTAGGGAAAGCAGGAGCCTCTCTGACCTTTAGCAGGGTCAGGGCCCCTCACCTTCCCCTCTTTTCCCAGAGCCGTCTTCCCAGCCCACCATCCCCATCGTGGGCATCATTGCTGGCCTGGTTCTCTTTGGAGCTGTGATCACTGGAGCTGTGGTCGCTGCTGTGATGTGGAGGAGGAAGAGCTCAGGTGGGGAAGGGGTGAAGGGTGGGTCTGAGATTTCTTGTCTCACTGAGGGTTCCAAGACCCAGGTAGAAGTGTGCCCTGCCTCGTTACTGGGAAGCACCACCCACAATTATGGGCCTACCCAGCCTGGGCCCTGTGTGCCAGCACTTACTCTTTTGTAAAGCACCTGTTAAAATGAAGGACAGATTTATCACCTTGATTACAGCGGTGATGGGACCTGATCCCAGCAGTCACAAGTCACAGGGGAAGGTCCCTGAGGACCTTCAGGAGGGCGGTTGGTCCAGGACCCACACCTGCTTTCTTCATGTTTCCTGATCCCGCCCTGGGTCTGCAGTCACACATTTCTGGAAACTTCTCTGAGGTCCAAGACTTGGAGGTTCCTCTAGGACCTTAAGGCCCTGACTCCTTTCTGGTATCTCACAGGACATTTTCTTCCCACAGATAGAAAAGGAGGGAGCTACTCTCAGGCTGCAAGTAAGTATGAAGGAGGCTGATGCCTGAGGTCCTTGGGATATTGTGTTTGGGAGCCCATGGGGGAGCTCACCCACCCCACAATTCCTCCTCTAGCCACATCTTCTGTGGGATCTGACCAGGTTCTGTTTTTGTTCTACCCCAGGCAGTGACAGTGCCCAGGGCTCTGATGTGTCTCTCACAGCTTGTAAAGGTGAGAGCCTGGAGGGCCTGATGTGTGTTGGGTGTTGGGCGGAACAGTGGACACAGCTGTGCTATGGGGTTTCTTTCCATTGGATGTATTGAGCATGCGATGGGCTGTTTAAAGTGTGACCCCTCACTGTGACAGATACGAATTTGTTCATGAATATTTTTTTCTATAGTGTGAGACAGCTGCCTTGTGTGGGACTGAGAGGCAAGAGTTGTTCCTGCCCTTCCCTTTGTGACTTGAAGAACCCTGACTTTGTTTCTGCAAAGGCACCTGCATGTGTCTGTGTTCGTGTAGGCATAATGTGAGGAGGTGGGGAGACCACCCCACCCCCATGTCCACCATGACCCTCTTCCCACGCTGACCTGTGCTCCCTCCCCAATCATCTTTCCTGTTCCAGAGAGGTGGGGCTGAGGTGTCTCCATCTCTGTCTCAACTTCATGGTGCACTGAGCTGTAACTTCTTCCTTCCCTATTAAAA"
exon5 = HLAA2_full_genomic[2265:2382]
exon5_nuc = "AGCCGTCTTCCCAGCCCACCATCCCCATCGTGGGCATCATTGCTGGCCTGGTTCTCTTTGGAGCTGTGATCACTGGAGCTGTGGTCGCTGCTGTGATGTGGAGGAGGAAGAGCTCAG"
IMGT = "VGIIAGLVLFGAVITGAVVAAVMW"
uniprot = "VGIIAGLVLLGAVITGAVVAAVMW"

In [51]:
exon5 == exon5_nuc

True

In [50]:
from Bio.Seq import Seq
ex5 = HLAA2_full_genomic[2265:2382]          # 1-based 2266–2382
print(Seq(exon5_nuc[2:-1]).translate())

PSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSS


In [ ]:
# reading frame shifted
ex5[2:-1]

'CCGTCTTCCCAGCCCACCATCCCCATCGTGGGCATCATTGCTGGCCTGGTTCTCTTTGGAGCTGTGATCACTGGAGCTGTGGTCGCTGCTGTGATGTGGAGGAGGAAGAGCTCA'

In [ ]:
# comparisons between sequences
import difflib
ref = IMGT
qry = uniprot
sm = difflib.SequenceMatcher(None, ref, qry, autojunk=False)
for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag != "equal":
        print(f"{tag:7s} ref[{i1}:{i2}]={ref[i1:i2]!r}  qry[{j1}:{j2}]={qry[j1:j2]!r}")

replace ref[9:10]='F'  qry[9:10]='L'


In [45]:
from Bio.Seq import Seq
print(Seq(exon5).translate())

SRLPSPPSPSWASLLAWFSLEL*SLELWSLL*CGGGRAQ


### Testing translation codon

EnforceGCContent:
- makes GC constraint local within the 50 nucleotide window such that each window will fall between 30-70% GC content

In [ ]:
import numpy as np
from numpy import random

def convert_nucleotide(aa_seq, seed=None):
    if seed is None:
        seed = random.randint(0,10000)

In [ ]:
import numpy as np
from dnachisel import (DnaOptimizationProblem, CodonOptimize,
                       EnforceTranslation, AvoidPattern, EnforceGCContent)
from dnachisel.biotools import reverse_translate, translate


def convert_nucleotide(aa_seq, seed, species="h_sapiens",
                       avoid_patterns=None,
                       gc_mini=0.2, gc_maxi=0.4, gc_window=50):
    """Codon-optimise one or several amino acid sequences into nucleotides.

    dnachisel draws all of its randomness from numpy's global RNG (both the
    randomised back-translation and the optimisation itself), so seeding that
    RNG is what makes a run reproducible. The seed is therefore required: no
    sequence is generated without one, so every sequence on record can be
    regenerated by calling this again with the same seed and the same inputs.

    Parameters
    ----------
    aa_seq
        A single amino acid sequence, a list of them, or a dict mapping a
        name to a sequence (e.g. leader_peptide_aa).
    seed
        Required int, used to seed numpy's global RNG once for the whole
        call. In a batch the sequences are generated from one continuing
        random stream, so reproducing any of them means rerunning the same
        call, not just the one sequence.
    species
        Species for the CodonOptimize objective.
    avoid_patterns
        Optional patterns to forbid. Accepts a single pattern ("BsaI_site")
        or a list of them. If None, no patterns are avoided.
    gc_mini, gc_maxi, gc_window
        Local GC content constraint, applied within each sliding window.

    Returns
    -------
    dna
        The optimised nucleotide sequence(s), matching the shape of aa_seq:
        a string for a single sequence, a list for a list, a dict for a dict.
    """
    if seed is None:
        raise ValueError(
            "a seed is required: the sequence cannot be reproduced without "
            "one, so pass an explicit seed (e.g. seed=1234) before running. "
            "Draw one with int(np.random.randint(0, 1_000_000)) and record it."
        )
    np.random.seed(int(seed))

    if avoid_patterns is None:
        avoid_patterns = []
    elif isinstance(avoid_patterns, str):
        avoid_patterns = [avoid_patterns]

    # Normalise the input to a (keys, sequences) pair, remembering its shape
    # so the results can be handed back in the same form.
    if isinstance(aa_seq, str):
        keys, sequences, shape = [None], [aa_seq], "single"
    elif isinstance(aa_seq, dict):
        keys, sequences, shape = list(aa_seq), list(aa_seq.values()), "dict"
    else:
        sequences = list(aa_seq)
        keys, shape = list(range(len(sequences))), "list"
    sequences = ["".join(s.split()).upper() for s in sequences]

    results = []
    for protein in sequences:
        problem = DnaOptimizationProblem(
            sequence=reverse_translate(protein, randomize_codons=True),
            constraints=[EnforceTranslation(),
                         *[AvoidPattern(pattern) for pattern in avoid_patterns],
                         EnforceGCContent(mini=gc_mini, maxi=gc_maxi,
                                          window=gc_window)],
            objectives=[CodonOptimize(species=species)],
            logger=None,
        )
        problem.resolve_constraints()
        problem.optimize()

        dna = problem.sequence
        assert translate(dna) == protein, \
            "translation does not match input protein"
        results.append(dna)

    if shape == "single":
        return results[0]
    if shape == "dict":
        return dict(zip(keys, results))
    return results


In [ ]:
convert_nucleotide(
    leader_peptide_aa,
    seed=123)

Created function is able to output nucleotide sequence

### Creating nucleotide sequence

Section will then create a standard nucleotide sequence for whatever aa input.
If input is a dictionary, will iterate and replace the value with 

Chain joins as:
5addon-SP-Peptide-L1-B2M-L2-HLA-3addon

In [ ]:
seed = 1231

SP = convert_nucleotide(
    leader_peptide_aa,
    seed=seed)

In [ ]:
seq = [SP['B2M'], peptide_nuc['MART-1'], linker_nuc["L1"], linker_nuc["L2"], HLA_nuc['HLA-A2']]
to_order = "".join(seq)
to_order

In [ ]:
to_order_all = {
    name: "".join([SP['B2M'], peptide, linker_nuc["L1"], linker_nuc["L2"], HLA_nuc['HLA-A2']])
    for name, peptide in peptide_nuc.items()
}
to_order_all

### Convert sequences into amino acid 

In [4]:
B2M_nuc = "atacaaagaactccaaagatccaagtttacagtagacatcctgctgaaaacggtaaatctaatttcttgaactgttacgtctccggtttccacccaagtgatatagaagttgacttgttgaaaaatggtgaaagaatcgaaaaggttgaacattcagatttgtctttttctaaggactggtccttctatttgttgtactacacagaattcactccaactgaaaaggatgaatacgcttgcagagttaatcatgtaaccttgtctcaacctaaaatcgttaagtgggatagagacatg"


In [10]:
from Bio.Seq import Seq

B2M_aa = Seq(B2M_nuc).translate()
print(B2M_aa)

IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM


In [6]:
B2M_full = "MSRSVALAVLALLSLSGLEAIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"

In [8]:
# if different use this to find different areas
# comparisons between sequences
import difflib
ref = B2M_full
qry = B2M_aa
sm = difflib.SequenceMatcher(None, ref, qry, autojunk=False)
for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag != "equal":
        print(f"{tag:7s} ref[{i1}:{i2}]={ref[i1:i2]!r}  qry[{j1}:{j2}]={qry[j1:j2]!r}")
    else:
        print("Both strings are equal!")

delete  ref[0:20]='MSRSVALAVLALLSLSGLEA'  qry[0:0]=Seq('')
Both strings are equal!
